# Fact Device Logs

## Overview
Creates **fact_device_logs** fact table containing device telemetry metrics and health indicators.

**Source**: device_logs (silver)
**Dimensions**: dim_device, dim_time
**Target**: telecom_catalog.gold_schema.fact_device_logs
**Partition**: event_date

---

## Step 1: Create Fact Table
Define partitioned fact table with device_key, time_key (FKs), metrics (CPU, memory, latency, temp), and health status fields.

In [0]:
CREATE OR REPLACE TABLE telecom_catalog.gold_schema.fact_device_logs
    (
    device_key BIGINT NOT NULL,
    time_key INT NOT NULL,

    cpu_usage DOUBLE,
    memory_usage DOUBLE,
    packet_loss INT,
    latency_ms INT,
    temp DOUBLE,

    status STRING,
    cpu_status STRING,
    memory_status STRING,
    network_health STRING,
    latency_category STRING,
    temperature_status STRING,
    overall_health_score STRING,

    event_date DATE
)
USING DELTA
PARTITIONED BY (event_date);

---
## Step 2: Prepare Source View
Join device_logs with dim_device (on device_id) and dim_time (on event_time date).

In [0]:
CREATE OR REPLACE TEMP VIEW vw_fact_device_logs_source AS

SELECT

    dd.device_key,
    dt.time_key,

    dl.cpu_usage,
    dl.memory_usage,
    dl.packet_loss,
    dl.latency_ms,
    dl.temp,

    dl.status,
    dl.cpu_status,
    dl.memory_status,
    dl.network_health,
    dl.latency_category,
    dl.temperature_status,
    dl.overall_health_score,

    TO_DATE(dl.event_time) AS event_date

FROM telecom_catalog.silver_schema.device_logs dl

INNER JOIN telecom_catalog.gold_schema.dim_device dd
    ON dl.device_id = dd.device_id

INNER JOIN telecom_catalog.gold_schema.dim_time dt
    ON TO_DATE(dl.event_time) = dt.full_date;

---
## Step 3: Validate Source Count
Verify source view record count.

In [0]:
SELECT COUNT(*)
FROM vw_fact_device_logs_source;

---
## Step 4: Check Foreign Keys
Validate no NULL values in device_key or time_key.

In [0]:
SELECT
    SUM(CASE WHEN device_key IS NULL THEN 1 ELSE 0 END) AS null_device_key,
        SUM(CASE WHEN time_key IS NULL THEN 1 ELSE 0 END) AS null_time_key
        FROM vw_fact_device_logs_source;

---
## Step 5: Load Fact Table
Insert all records from source view into partitioned fact table.

In [0]:
INSERT INTO telecom_catalog.gold_schema.fact_device_logs
SELECT *
FROM vw_fact_device_logs_source;

---
## Step 6: Validate Load
Verify final record count after insert.

In [0]:
SELECT COUNT(*)
FROM telecom_catalog.gold_schema.fact_device_logs;